### py4cytoscape

https://py4cytoscape.readthedocs.io/en/latest/

In [ ]:
from pathlib import Path
from rdflib import Graph, RDF, Namespace

import os
import json
import math

import dash
from dash import html, Input, Output, State, dcc
import dash_cytoscape as cyto
import dash_bootstrap_components as dbc

import networkx as nx
import py4cytoscape as p4c


### Cytoscape 

- run

> export INSTALL4J_JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64  
> ~/Cytoscape_v3.10.2/Cytoscape &


In [ ]:
root_owl = Path('../owl/')
BP = Namespace("http://www.biopax.org/release/biopax-level3.owl#")

G = nx.DiGraph()

classes = [
    BP.Pathway,
    BP.BiochemicalReaction,
    BP.Catalysis,
    BP.Control,
    BP.Protein,
    BP.Complex,
    BP.SmallMolecule,
    BP.PhysicalEntity,
]

relations = [
    BP.pathwayComponent,
    BP.left,
    BP.right,
    BP.controller,
    BP.controlled,
    BP.participant,
    BP.component,
]


def read_owl(owl_file):
    fnameowl = root_owl / owl_file

    rdf = Graph()
    rdf.parse(fnameowl, format="xml")

    return rdf

def short(x):
    return str(x).split("#")[-1].split("/")[-1]

def get_name(x, rdf):
    for prop in [BP.displayName, BP.standardName, BP.name]:
        value = next(rdf.objects(x, prop), None)
        if value:
            return str(value)
    return short(x)

In [ ]:
owl_file = "R-HSA-165159_level3.owl"
owl_file = "R-HSA-114608_level3.owl"

rdf = read_owl(owl_file)

pathway_name = 'Platelet degranulation'
pathway_id = "R-HSA-114608"

owl_file = f"{pathway_id}_level3.owl"

rdf = read_owl(owl_file)

In [ ]:
for cls in classes:
    for node in rdf.subjects(RDF.type, cls):
        node_id = short(node)
        G.add_node(node_id, label=get_name(node, rdf), biopax_type=short(cls))

In [ ]:
for rel in relations:
    for s, o in rdf.subject_objects(rel):
        s_id = short(s)
        o_id = short(o)

        if s_id in G.nodes and o_id in G.nodes:
            G.add_edge(s_id, o_id, interaction=short(rel))

```Python
G.nodes[gene_id]["log2FC"] = log2fc
G.nodes[gene_id]["FDR"] = fdr
G.nodes[gene_id]["mutated"] = True
```

### OWL to Graph

In [ ]:
fname_pos = "positions_%s.json"

def load_positions(pathway_id:str):

    fname=fname_pos%(pathway_id)
    filename = root_owl / fname

    try:
        if os.path.exists(filename):
            with open(filename) as f:
                print(f"Data loaded from {filename}")
                return json.load(f)
    except Exception as e:
        print(f"Error loading positions from {filename}: {e}")
    return {}

In [ ]:
global saved_positions

saved_positions = load_positions(pathway_id)

In [ ]:

def nx_to_cytoscape_elements(G):
    elements = []

    global saved_positions
    saved_positions = load_positions(pathway_id)

    missing = set(saved_positions.keys()) - set(G.nodes())
    if missing:
        print(f"⚠️ positions for {len(missing)} nodes not used (graph changed)")
        for node_id in missing:
            print(f"  - {node_id}")

    for node_id, data in G.nodes(data=True):
        elem = {
            "data": {
                "id": node_id,
                "label": data.get("label", node_id),
                "biopax_type": data.get("biopax_type", "Unknown"),
                "log2FC": data.get("log2FC", None),
                "FDR": data.get("FDR", None),
            }
        }

        if node_id in saved_positions:
            elem["position"] = saved_positions[node_id]

        elements.append(elem)

    for source, target, data in G.edges(data=True):
        elements.append({
            "data": {
                "source": source,
                "target": target,
                "interaction": data.get("interaction", "")
            }
        })

    return elements



In [ ]:
# remove nodes with degree 0

# isolated_nodes = list(nx.isolates(G))
# G.remove_nodes_from(isolated_nodes)

G = G.subgraph(
    [n for n in G.nodes() if G.degree(n) > 0]
).copy()

elements = nx_to_cytoscape_elements(G)

### Dash

In [ ]:
G = G.subgraph(
    [n for n in G.nodes() if G.degree(n) > 0]
).copy()

elements = nx_to_cytoscape_elements(G)

app = dash.Dash(__name__)

title = f"{pathway_name} - {pathway_id}"

height = "900px"
width = "100%"

app.layout = html.Div([
    html.H2(
        title,
        style={
            "textAlign": "center",
            "color": "#333",
            "marginTop": "5px",
            "marginBottom": "5px"
        }
    ),

    cyto.Cytoscape(
        id="reactome-network",
        elements=elements,
        layout={"name": "cose"},
        style={"width": width, 
               "height": height,
               "backgroundColor": "#fff9c4",  # light yellow
               },
        stylesheet=[
            {
                "selector": "node",
                "style": {
                    "label": "data(label)",
                    "font-size": "10px",
                    "background-color": "#8ecae6",
                    "width": 30,
                    "height": 30,
                    "marginTop": "60px",
                },
            },
            {
                "selector": "edge",
                "style": {
                    "curve-style": "bezier",
                    "target-arrow-shape": "triangle",
                    "label": "data(interaction)",
                    "font-size": "7px",
                },
            },
            {
                "selector": '[biopax_type = "BiochemicalReaction"]',
                "style": {"background-color": "#f94144", "shape": "hexagon"},
            },
            {
                "selector": '[biopax_type = "Protein"]',
                "style": {"background-color": "#8ecae6", "shape": "ellipse"},
            },
            {
                "selector": '[biopax_type = "Complex"]',
                "style": {"background-color": "#ffb703", "shape": "round-rectangle"},
            },
            {
                "selector": '[biopax_type = "SmallMolecule"]',
                "style": {"background-color": "#90be6d", "shape": "diamond"},
            },
            {
                "selector": '[biopax_type = "Pathway"]',
                "style": {"background-color": "#cdb4db", "shape": "rectangle"},
            },
        ],
    ),

    html.Button("Save node positions", id="save-button"), html.Pre(id="saved-output"),

    dbc.Toast(
        id="save-toast",
        header="Success",
        is_open=False,
        dismissable=True,
        duration=2500,
        icon="success",
        style={
            "position": "fixed",
            "top": 20,
            "right": 20,
            "width": 360,
            "zIndex": 9999,
        },
    )
])

In [ ]:
def extract_positions(elements):
    return {
        e["data"]["id"]: e["position"]
        for e in elements
        if "data" in e and "id" in e["data"] and "position" in e
    }

def positions_changed(new, tol=1e-3):

    if saved_positions.keys() != new.keys():
        print("Error: different node sets.")
        return True

    for k in saved_positions:
        if not (
            math.isclose(saved_positions[k]["x"], new[k]["x"], abs_tol=tol) and
            math.isclose(saved_positions[k]["y"], new[k]["y"], abs_tol=tol)
        ):
            return True

    return False

def save_positions_if_changed(elements) -> bool:
    global saved_positions
    new_pos = extract_positions(elements)

    if not positions_changed(new_pos):
        return False  # no change

    fname=fname_pos%(pathway_id)
    filename = root_owl / fname

    try:
        with open(filename, "w") as f:
            json.dump(new_pos, f, indent=2)

        saved_positions = new_pos  # update global here
    except Exception as e:
        print(f"Error: saving Graph positions: {e}")

    return True

@app.callback(
    Output("save-toast", "is_open"),
    Output("save-toast", "children"),
    Input("save-button", "n_clicks"),
    State("reactome-network", "elements"),
    prevent_initial_call=True,
)
def save_and_notify(n_clicks, elements):

    changed = save_positions_if_changed(elements)

    if changed:
        message = f"Graph saved for: {pathway_name} - {pathway_id}"
    else:
        message = f"No changes detected"

    return True, message

In [ ]:
app.run(debug=True, port=8050)